### Submission 3: Parse remote content during change of Mind
#### Insert into table ChangeofMindTheta

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import pickle
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import ranksums
import seaborn as sns
import copy

In [3]:
from spyglass.shijiegu.Analysis_SGU import get_linearization_map

from spyglass.shijiegu.Analysis_SGU import TrialChoice, DecodeIngredients, DecodeResults2D, ChangeofMind, ChangeofMindRemoteTheta, DecodeResultsLinear, ChangeofMindTriggeredDecode
from spyglass.shijiegu.decodeHelpers import runSessionNames
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from spyglass.decoding.v0.visualization import make_single_environment_movie
from spyglass.shijiegu.changeOfMind_triggered_position import load_triggered_position_decode_day
from spyglass.shijiegu.changeOfMind_triggered import select_subset_helper_position
from spyglass.shijiegu.changeOfMind_remote import find_posterior_sum_segment, find_remote_theta_animal_new
from spyglass.shijiegu.changeOfMind_remote_interval import find_remote_theta_animal, parse_remote_master, load_remote_animal

[2026-04-03 10:27:58,753][INFO]: DataJoint 0.14.4 connected to shijiegu-alt@lmf-db.cin.ucsf.edu:3306
[10:28:12][WARNING] Spyglass: Deprecation: this class has been moved out of spyglass.common.common_position
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
[10:28:12][WARNING] Spyglass: Deprecation: this class has been moved out of spyglass.common.common_position
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
[10:28:12][WARNING] Spyglass: Deprecation: this class has been moved out of spyglass.common.common_position
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
	TrackGraph -> spyglass.linearization.v0.main.TrackGraph
Please use the new location.
[10:28:13][WARNING] Spyglass: Deprecation: this class has b

In [4]:
ChangeofMindTriggeredDecode() & {"nwb_file_name": "julio20230801_.nwb"}#"klein20231108_.nwb", "parameter":"params_both_control_max_segment_run_time_2_state"}#"julio20230811_.nwb"}#"parameter":"params_both_max_run_time_2_state"}#"parameter":"params_both_max_segment_run_time_2_state"}


nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),proportion minimal amount of proportion the animal is in before it backed out,parameter parameter name,analysis_file_name name of the file,parameter_value parameter value
julio20230801_.nwb,2,0.1,params_both_control_max_segment_all_maze_2_state,julio20230801_C0YCRJ929E.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_both_control_max_segment_run_time_2_state,julio20230801_UKQNDCH8OQ.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_both_max_segment_all_maze_2_state,julio20230801_0ZTKRN8WAG.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_both_max_segment_run_time_2_state,julio20230801_1WB8S3JFKA.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_post_all_maze_2_state,julio20230801_926DAR6YBH.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_post_control_run_time_2_state,julio20230801_FUMRGZ773F.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_post_run_time_2_state,julio20230801_UHZOZ0X00F.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_pre_all_maze_2_state,julio20230801_E81QPM96DZ.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_pre_control_all_maze_2_state,julio20230801_1GVPU696ER.nwb,=BLOB=
julio20230801_.nwb,2,0.1,params_pre_control_run_time_2_state,julio20230801_XEI2S57TB5.nwb,=BLOB=


In [13]:
params = {"use_1d": True,
          "use_home": False,
          "use_center": False,
          "use_outer": True,
          "proportion":0.1,
          "parameter_name": "params_both_max_all_maze_2_state"}

In [14]:
list_of_days_animals = {}
success_animal = {}

animal = "eliot"
list_of_days_animals[animal] = ['20221018','20221019','20221020','20221021','20221022','20221023','20221024','20221025','20221026']

animal = "molly"
list_of_days_animals[animal]  = ['20220416','20220417','20220418','20220419','20220420']

animal = 'lewis'
list_of_days_animals[animal] = ['20240105','20240106','20240107','20240108','20240109',
                 '20240110']

animal = 'julio'
list_of_days_animals[animal] = ['20230731','20230801','20230802','20230803','20230804',
                 '20230805','20230806','20230807','20230808','20230809','20230810','20230811']

animal = 'klein'
list_of_days_animals[animal]  = ['20231101','20231102','20231103','20231104','20231105',
                 '20231106','20231107','20231108']

### 2. debug start

In [20]:
log_df = pd.DataFrame((ChangeofMindRemoteTheta() & {"nwb_file_name": "molly20220418_.nwb","epoch":10,
                                                    "parameter":"params_both_max_all_maze_2_state"}).fetch1("pandas"))
log_df_com = log_df[log_df.has_remote_interval]
log_df_com

,timestamp_H,Home,timestamp_O,OuterWellIndex,rewardNum,change_of_mind,has_remote_interval,remote_interval,remote_content,change_of_mind_num,...,proportion_arm3,proportion_arm4,CoMNum_by_time,CoMNum_by_arm,CoM_arm,current,future_H,future_O,past,past_reward
28,1.650325e+09,1.0,1.650325e+09,4.0,1.0,True,True,"[[1650325175.3036144, 1650325175.3856144], [16...","[0, 0, 0, 0]","[0, 0, 0, 0]",...,0.584492,0.000000,1,1,"[[3, 4, -9223372036854775808]]",4.0,4.0,1.0,2.0,2.0
34,1.650325e+09,1.0,1.650325e+09,4.0,1.0,True,True,"[[1650325328.8835862, 1650325328.947586], [165...","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]",...,0.000000,0.000000,2,2,"[[2, 1, 4]]",4.0,4.0,3.0,1.0,1.0
39,1.650325e+09,1.0,1.650325e+09,3.0,1.0,True,True,"[[1650325473.2475595, 1650325473.2775595]]",[1],[0],...,0.000000,0.360574,1,1,"[[4, 3, -9223372036854775808]]",3.0,3.0,1.0,2.0,2.0


In [21]:
log_df = pd.DataFrame((ChangeofMindRemoteTheta() & {"nwb_file_name": "molly20220418_.nwb","epoch":10,
                                                    "parameter":"params_both_max_run_time_2_state"}).fetch1("pandas"))
log_df_com = log_df[log_df.has_remote_interval]
log_df_com

,timestamp_H,Home,timestamp_O,OuterWellIndex,rewardNum,change_of_mind,has_remote_interval,remote_interval,remote_content,change_of_mind_num,...,proportion_arm3,proportion_arm4,CoMNum_by_time,CoMNum_by_arm,CoM_arm,current,future_H,future_O,past,past_reward
17,1.650325e+09,1.0,1.650325e+09,2.0,2.0,True,True,"[[1650324943.373657, 1650324943.399657], [1650...","[0, 0]","[0, 0]",...,0.000000,0.626893,2,2,"[[4, 1, 2, -9223372036854775808]]",2.0,2.0,4.0,1.0,4.0
28,1.650325e+09,1.0,1.650325e+09,4.0,1.0,True,True,"[[1650325175.3416142, 1650325175.3916142]]",[4],[0],...,0.584492,0.000000,1,1,"[[3, 4, -9223372036854775808]]",4.0,4.0,1.0,2.0,2.0
39,1.650325e+09,1.0,1.650325e+09,3.0,1.0,True,True,"[[1650325473.2475595, 1650325473.2775595]]",[1],[0],...,0.000000,0.360574,1,1,"[[4, 3, -9223372036854775808]]",3.0,3.0,1.0,2.0,2.0


### 2. debug end

In [15]:
params = {"use_1d": True,
          "use_home": False,
          "use_center": False,
          "use_outer": True,
          "proportion":0.1,
          "parameter_name": "params_both_max_all_maze_2_state"}#"params_both_max_run_time_2_state"} #"params_both_max_all_maze_2_state", "params_both_max_run_time_2_state"

success_animal = {}
for animal in ["lewis","eliot","molly","klein","julio"]:#list_of_days_animals.keys():
    days = list_of_days_animals[animal]
    success = parse_remote_master(animal,
                      days,
                      params,
                      minimum_duration = 0.02,
                      min_sum_posterior = 0.2,
                      fill_spyglass = True)

    success_animal[animal] = success

[2026-01-25 20:40:33,503][WARNING]: Skipped checksum for file with hash: b13385bc-86f6-f0fa-922b-9e4a5ccea5fb, and path: /stelmo/nwb/analysis/lewis20240105/lewis20240105_CTI07FVT1J.nwb


lewis20240105_.nwb
*nwb_file_name *epoch    *proportion    analysis_file_ pandas_id     pandas    
+------------+ +-------+ +------------+ +------------+ +-----------+ +--------+
lewis20240105_ 2         0.1                                         =BLOB=    
 (Total: 1)

Found remote theta in trial  [42, 42, 42]
Found remote theta in trial  [46]
Found remote theta in trial  [50, 50, 50, 50, 50, 50]
Found remote theta in trial  [62]
Found remote theta in trial  [65]
Found remote theta in trial  [66]
Found remote theta in trial  [70, 70, 70, 70, 70, 70, 70, 70]
Found remote theta in trial  [71, 71]
Found remote theta in trial  [88]


[2026-01-25 20:40:50,669][WARNING]: Skipped checksum for file with hash: b3b4c94a-5c8c-19c9-fd43-efe78342ac39, and path: /stelmo/nwb/analysis/lewis20240105/lewis20240105_9YD57CSAZY.nwb


Found remote theta in trial  [98, 98, 98, 98, 98, 98, 98]
*nwb_file_name *epoch    *proportion    analysis_file_ pandas_id     pandas    
+------------+ +-------+ +------------+ +------------+ +-----------+ +--------+
lewis20240105_ 4         0.1                                         =BLOB=    
 (Total: 1)

Found remote theta in trial  [5]
Found remote theta in trial  [9, 9]
Found remote theta in trial  [16]
Found remote theta in trial  [19]
Found remote theta in trial  [22, 22, 22]
Found remote theta in trial  [30, 30, 30]
Found remote theta in trial  [43]
Found remote theta in trial  [77, 77]


[2026-01-25 20:41:04,504][WARNING]: Skipped checksum for file with hash: 40e08994-024e-5e63-f8ff-f2b513dba795, and path: /stelmo/nwb/analysis/lewis20240105/lewis20240105_NLFK1MPAM4.nwb


...

julio20230810_.nwb
No triggered decode found for  {'nwb_file_name': 'julio20230810_.nwb', 'epoch': 2, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
*nwb_file_name *epoch    *proportion    analysis_file_ pandas_id     pandas    
+------------+ +-------+ +------------+ +------------+ +-----------+ +--------+
julio20230810_ 4         0.1                                         =BLOB=    
 (Total: 1)

Found remote theta in trial  [31, 31]
Found remote theta in trial  [49]
Found remote theta in trial  [51, 51, 51, 51]


[2026-01-25 21:11:04,780][WARNING]: Skipped checksum for file with hash: 72f4211e-adf0-6c73-8d5a-afdf050eb28c, and path: /stelmo/nwb/analysis/julio20230810/julio20230810_SQ6CRIDB28.nwb


*nwb_file_name *epoch    *proportion    analysis_file_ pandas_id     pandas    
+------------+ +-------+ +------------+ +------------+ +-----------+ +--------+
julio20230810_ 6         0.1                                         =BLOB=    
 (Total: 1)

Found remote theta in trial  [12, 12, 12]


[2026-01-25 21:11:15,864][WARNING]: Skipped checksum for file with hash: f395c122-ded6-2d5d-9dc3-66f3411d5263, and path: /stelmo/nwb/analysis/julio20230811/julio20230811_HC4TIPN17N.nwb


Found remote theta in trial  [68, 68]
No triggered decode found for  {'nwb_file_name': 'julio20230810_.nwb', 'epoch': 8, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
julio20230811_.nwb
No triggered decode found for  {'nwb_file_name': 'julio20230811_.nwb', 'epoch': 2, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
No triggered decode found for  {'nwb_file_name': 'julio20230811_.nwb', 'epoch': 4, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
No triggered decode found for  {'nwb_file_name': 'julio20230811_.nwb', 'epoch': 6, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
No triggered decode found for  {'nwb_file_name': 'julio20230811_.nwb', 'epoch': 7, 'proportion': 0.1, 'parameter': 'params_both_max_all_maze_2_state'}
*nwb_file_name *epoch    *proportion    analysis_file_ pandas_id     pandas    
+------------+ +-------+ +------------+ +------------+ +-----------+ +--------+
julio20230811_ 9         0.1

In [ ]:
success_animal

In [11]:
success_animal

{'lewis': 1, 'eliot': 1, 'molly': 1, 'klein': 1, 'julio': 1}

In [23]:
ChangeofMindRemoteTheta() & {"minimum_duration": 0.02}

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),proportion minimal amount of proportion the animal is in before it backed out,parameter parameter name,"minimum_duration time in seconds, the min duration of an event","pandas pandas dataframe saved as dictionary, choice"
eliot20221018_.nwb,2,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221018_.nwb,5,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221018_.nwb,7,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221018_.nwb,9,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221018_.nwb,11,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221019_.nwb,2,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221019_.nwb,4,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221019_.nwb,6,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221019_.nwb,8,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
eliot20221019_.nwb,10,0.1,params_both_max_run_time_2_state,0.02,=BLOB=
